In [2]:
import pandas as pd
import numpy as np
import os

## Define Feature Exaction

In [ ]:
def extract_features(data_window):
    """
    Takes a 2-second window (approx 100 samples) and returns 5 key features.
    """
    # Calculate Magnitude R = sqrt(x^2 + y^2 + z^2)
    # We do this for both Accel and Gyro
    accel_mag = np.sqrt(data_window['acc_x']**2 + data_window['acc_y']**2 + data_window['acc_z']**2)
    gyro_mag = np.sqrt(data_window['gyro_x']**2 + data_window['gyro_y']**2 + data_window['gyro_z']**2)
    
    features = [
        np.mean(accel_mag),        # Overall intensity
        np.std(accel_mag),         # Jitter/Erraticness
        np.max(accel_mag),         # Sudden spikes (Braking/Impact)
        np.mean(gyro_mag),         # Rotation intensity
        np.std(gyro_mag)           # Swerving consistency
    ]
    return features

def process_dataset(directory_path, label_map, window_size=100):
    X, y = [], []
    
    # Iterate through folders (Normal, Aggressive, Risky)
    for label_name, label_value in label_map.items():
        folder_path = os.path.join(directory_path, label_name)
        if not os.path.exists(folder_path): continue
            
        for file in os.listdir(folder_path):
            if file.endswith(".csv"):
                df = pd.read_csv(os.path.join(folder_path, file))
                # Slide through the CSV in windows
                for i in range(0, len(df) - window_size, window_size // 2): # 50% overlap
                    window = df.iloc[i:i+window_size]
                    X.append(extract_features(window))
                    y.append(label_value)
                    
    return np.array(X), np.array(y)

## Data IMport

In [8]:
def get_features(window):
    # Vector Magnitude for orientation-independence
    acc_mag = np.sqrt(window['X']**2 + window['Y']**2 + window['Z']**2)
    gyro_mag = np.sqrt(window['X']**2 + window['Y']**2 + window['Z']**2)
    
    return [
        acc_mag.mean(), 
        acc_mag.std(), 
        acc_mag.max(), 
        gyro_mag.mean(), 
        gyro_mag.std()
    ]

In [9]:
from sklearn.ensemble import RandomForestClassifier

X, y = [], []
base_path = "./datasetAccel"
labels = {'Normal': 0, 'Dangerous': 1}

for label_name, val in labels.items():
    folder = os.path.join(base_path, label_name)
    for file in os.listdir(folder):
        df = pd.read_csv(os.path.join(folder, file))
        # Windowing: 100 samples = 2 seconds at 50Hz
        for i in range(0, len(df)-100, 50): 
            X.append(get_features(df.iloc[i:i+100]))
            y.append(val)

model = RandomForestClassifier(n_estimators=50).fit(X, y)
print("Brain Trained!")

Brain Trained!
